# 3.2 Logistic Regression

**Class 4:** which factors are associated with the **odds of having diabetes**?

1. Look at the outcome
2. **Univariable** models: one predictor at a time
3. **Multivariable** models: add predictors in blocks and watch how the odds ratios change

Logistic regression models a **yes/no outcome** (here `diabetes_binary`: 1 = diabetes, 0 = no diabetes). Results are reported as **odds ratios (OR)**.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data or counts of 1 to 20. Clear outputs before you share or commit this notebook.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

In [ ]:
library(broom)   # tidy() turns model output into a data frame

fa <- prepare_analysis_data()

## Look at the outcome

Before modelling, check that the outcome is coded 0/1 and that there are enough people in each group.

In [ ]:
fa %>% count(diabetes_binary)

## Univariable (crude) models

`glm(..., family = binomial)` fits a logistic regression. `tidy(exponentiate = TRUE, conf.int = TRUE)` reports **odds ratios** with 95% confidence intervals.

- **OR > 1:** higher odds of diabetes
- **OR < 1:** lower odds
- **OR = 1:** no association

For a continuous predictor (age, BMI), the OR is per **1-unit** increase.

In [ ]:
univariable <- function(predictor) {
  glm(reformulate(predictor, "diabetes_binary"), data = fa, family = binomial) %>%
    tidy(exponentiate = TRUE, conf.int = TRUE) %>%
    filter(term != "(Intercept)")
}

map_dfr(c("age", "BMI", "gender", "ptsd_doctor", "ptsd_treatment"), univariable) %>%
  select(term, estimate, conf.low, conf.high, p.value)

**Interpret:** for each predictor, write one sentence. For example: *"Each 1-unit increase in BMI is associated with X% higher odds of diabetes (95% CI …)."* (An OR of 1.06 means 6% higher odds.)

These are **unadjusted**: they ignore every other variable.

## Multivariable models, step by step

We add predictors in blocks and keep each model:

| Model | Adds | Why |
|---|---|---|
| Base | age, gender, race, ethnicity | standard demographic confounders |
| +BMI | BMI | strong clinical risk factor |
| +BP+LDL | systolic BP, LDL cholesterol | cardiometabolic markers |
| Full | the two PTSD variables | our variables of interest |

`update()` adds terms to an existing formula: `. ~ . + BMI` means "keep everything, add BMI".

In [ ]:
form_base  <- diabetes_binary ~ age + gender + race + ethnicity
form_bmi   <- update(form_base, . ~ . + BMI)
form_bp    <- update(form_bmi,  . ~ . + Systolic_BP + LDL_Cholesterol)
form_full  <- update(form_bp,   . ~ . + ptsd_doctor_binary + ptsd_treatment_binary)

models <- list(
  "Base"    = glm(form_base, data = fa, family = binomial),
  "+BMI"    = glm(form_bmi,  data = fa, family = binomial),
  "+BP+LDL" = glm(form_bp,   data = fa, family = binomial),
  "Full"    = glm(form_full, data = fa, family = binomial)
)

## Compare the odds ratios across models

`imap_dfr()` (from **purrr**) runs the same code on every model in the list and stacks the results. `.x` is the model and `.y` is its name.

In [ ]:
imap_dfr(models, ~ tidy(.x, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = .y)) %>%
  filter(term %in% c("BMI", "Systolic_BP", "LDL_Cholesterol",
                     "ptsd_doctor_binary", "ptsd_treatment_binary")) %>%
  select(model, term, estimate, conf.low, conf.high, p.value) %>%
  arrange(term, model)

## Discuss

1. **Stability:** does the BMI odds ratio change much as we add variables? What does that suggest about confounding?
2. **Change in direction:** do either of the PTSD odds ratios move a lot, or cross 1, once other variables are included? What could explain that?
3. **Statistical vs clinical significance:** an OR of 0.996 per mg/dL can be "significant" but tiny. Is it meaningful?
4. **Write it in plain language:** *"After accounting for demographics, BMI, blood pressure, and LDL, people receiving PTSD treatment had X times the odds of diabetes (95% CI …)."*

**Going further:** `tidy(models$Full, exponentiate = TRUE)` shows every coefficient; `anova(models$Base, models$Full, test = "LRT")` compares nested models.